In [56]:
import pandas as pd
import json
import re
import os
from transformers import BertTokenizer

In [50]:


# Load the dataset
dataset_number = 1
with open(f'./Datasets/dataset_{dataset_number}_train.json') as file:
    raw_ds = json.load(file)

# Flatten JSON to DataFrame
df = pd.json_normalize(raw_ds)

# Step 1: Flatten JSON-like fields
def flatten_field(field):
    if isinstance(field, list):
        return ";".join(map(str, field))
    elif isinstance(field, dict):
        return json.dumps(field)
    return field

for col in df.columns:
    df[col] = df[col].apply(flatten_field)

# Step 2: Clean text fields and remove IP in `request.headers.Host`
def clean_text(text, column_name=None):
    if not isinstance(text, str):
        return text
    # Remove dangerous patterns, extra spaces, and non-alphanumeric characters
    text = re.sub(r'\${.*?}', '', text)  # Remove Log4j-style payloads
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    
    # Remove IP address in 'request.headers.Host' but keep the port
    if column_name == "request.headers.Host":
        text = re.sub(r'^\d{1,3}(\.\d{1,3}){3}', '', text).strip(":")
    return text

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(lambda x: clean_text(x, column_name=col))

# Step 3: Drop irrelevant columns
irrelevant_columns = [
    "request.headers.Date",  # Irrelevant for payload detection
    "response.status_code",  # Similar info already in `response.status`
]
df.drop(columns=irrelevant_columns, inplace=True, errors="ignore")

# Step 4: Ensure labels and payload text
if 'request.Attack_Tag' in df.columns:
    df.rename(columns={"request.Attack_Tag": "label"}, inplace=True)
if 'request.url' in df.columns:
    df.rename(columns={"request.url": "payload"}, inplace=True)

# Preview the modified DataFrame
print("\nModified DataFrame Preview:")
df.head()



Modified DataFrame Preview:


,request.headers.Host,request.headers.User-Agent,request.headers.Accept-Encoding,request.headers.Accept,request.headers.Connection,request.headers.Accept-Language,request.headers.Sec-Fetch-Site,request.headers.Sec-Fetch-Mode,request.headers.Sec-Fetch-User,request.headers.Sec-Fetch-Dest,...,request.body,label,response.status,response.headers.Content-Type,response.headers.Content-Length,response.body,request.headers.Cookie,response.headers.Location,request.headers.Content-Length,response.headers.Set-Cookie
0,5000,mozilla/5.0 (x11; linux i586; rv:31.0) gecko/2...,"gzip, deflate, br",*/*,keep-alive,de-ch,none,same-origin,?1,document,...,,directory traversal,200 ok,application/json,72,"{""error"": ""file ../../../../../../../../window...",NaN,NaN,NaN,NaN
1,5000,mozilla/5.0 (x11; openbsd amd64; rv:28.0) geck...,"gzip, deflate, br",*/*,keep-alive,de,none,same-origin,?1,document,...,,NaN,404 not found,application/json,41,"{""error"": ""category name not found""}",NaN,NaN,NaN,NaN
2,5000,mozilla/5.0 (x11; ubuntu; linux x86_64; rv:24....,"gzip, deflate, br",*/*,keep-alive,de,none,same-origin,?1,document,...,,cookie injection,200 ok,text/html; charset=utf-8,105,<h1>logged in as cedric</h1><form method='post...,username=gasvyqaaaaaaaacmcgj1awx0aw5zliwezxzhb...,NaN,NaN,NaN
3,5000,mozilla/5.0 (windows nt 6.1; rv:27.3) gecko/20...,"gzip, deflate, br",*/*,keep-alive,de-ch,none,same-origin,?1,document,...,,NaN,404 not found,application/json,30,"{""error"": ""not a region""}",NaN,NaN,NaN,NaN
4,5000,mozilla/5.0 (windows nt 6.1; win64; x64; rv:25...,"gzip, deflate, br",*/*,keep-alive,de,none,same-origin,?1,document,...,,log4j,404 not found,application/json,41,"{""error"": ""category name not found""}",NaN,NaN,NaN,NaN


In [51]:
# Check dataset statistics after cleaning
print("\nDataset Info After Preprocessing:")
df.info()


Dataset Info After Preprocessing:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4282 entries, 0 to 4281
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   request.headers.Host             4282 non-null   object
 1   request.headers.User-Agent       4282 non-null   object
 2   request.headers.Accept-Encoding  4282 non-null   object
 3   request.headers.Accept           4282 non-null   object
 4   request.headers.Connection       4282 non-null   object
 5   request.headers.Accept-Language  4282 non-null   object
 6   request.headers.Sec-Fetch-Site   4282 non-null   object
 7   request.headers.Sec-Fetch-Mode   4282 non-null   object
 8   request.headers.Sec-Fetch-User   4282 non-null   object
 9   request.headers.Sec-Fetch-Dest   4282 non-null   object
 10  request.headers.Set-Cookie       4282 non-null   object
 11  request.method                   4282 non-null   object
 12 

In [52]:
df.describe(include='all')

,request.headers.Host,request.headers.User-Agent,request.headers.Accept-Encoding,request.headers.Accept,request.headers.Connection,request.headers.Accept-Language,request.headers.Sec-Fetch-Site,request.headers.Sec-Fetch-Mode,request.headers.Sec-Fetch-User,request.headers.Sec-Fetch-Dest,...,request.body,label,response.status,response.headers.Content-Type,response.headers.Content-Length,response.body,request.headers.Cookie,response.headers.Location,request.headers.Content-Length,response.headers.Set-Cookie
count,4282,4282,4282,4282,4282,4282,4282,4282,4282,4282,...,4282,2264,4282,4282,4282,4282,566,401,299,299
unique,1,33,1,1,1,4,1,2,1,1,...,1,6,5,2,92,306,38,2,1,1
top,5000,mozilla/5.0 (windows nt 6.1; rv:27.3) gecko/20...,"gzip, deflate, br",*/*,keep-alive,de-ch,none,websocket,?1,document,...,,sql injection,200 ok,application/json,207,<!doctype html> <html lang=en> <title>404 not ...,username=gasvzgaaaaaaaacmcgj1awx0aw5zliwezxzhb...,/cookielogin,0,username=gasvkgaaaaaaaacmcf9fbwfpbl9fliwgugvyc...
freq,4282,160,4282,4282,4282,1086,4282,2165,4282,4282,...,4282,586,1958,2366,452,452,36,299,299,299


In [ ]:


# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Ensure the DataFrame has 'payload' and 'label' columns
if "payload" not in df.columns or "label" not in df.columns:
    raise ValueError("The DataFrame must contain 'payload' and 'label' columns.")

# Step 1: Preprocess the payload column to remove IPs
def remove_ip_from_payload(payload):
    if not isinstance(payload, str):
        return payload
    # Remove IP addresses but keep the port
    return re.sub(r'\b\d{1,3}(\.\d{1,3}){3}', '', payload).strip(":/")

df["payload"] = df["payload"].apply(remove_ip_from_payload)

# Step 2: Tokenize the payload texts and match with labels
tokenized_data = []
for _, row in df.iterrows():
    payload = row.get("payload")
    label = row.get("label")
    if pd.notnull(payload) and pd.notnull(label):
        tokens = tokenizer.tokenize(payload)  # Tokenize the cleaned payload
        tokenized_data.append({"tokens": tokens, "label": label})  # Add tokens and label

# Save the tokenized data into a `.dic` file format
output_file = './Datasets/tokenized_payloads.dic'
with open(output_file, 'w', encoding='utf-8') as outfile:
    json.dump(tokenized_data, outfile, ensure_ascii=False, indent=4)

# Print a preview of the tokenized data
print("\nTokenized Data Example:")
for example in tokenized_data[:5]:  # Show first 5 examples for clarity
    print(f"Tokens: {example['tokens']}")
    print(f"Label: {example['label']}\n")



Tokenized Data Example:
Tokens: ['http', ':', '/', '/', ':', '5000', '/', 'static', '/', 'download', '_', 'tx', '##t', '/', '.', '.', '/', '.', '.', '/', '.', '.', '/', '.', '.', '/', '.', '.', '/', '.', '.', '/', '.', '.', '/', '.', '.', '/', 'windows', '.', 'in', '##i', '.', 'tx', '##t']
Label: directory traversal

Tokens: ['http', ':', '/', '/', ':', '5000', '/', 'cookie', '##log', '##in']
Label: cookie injection

Tokens: ['http', ':', '/', '/', ':', '5000', '/', 'categories', '/', 'check', '/', 'name', '/', '250', '##8']
Label: log4j

Tokens: ['http', ':', '/', '/', ':', '5000', '/', 'log', '##in', '/', 'ad', '##min', '/', 'password', '/', '1980']
Label: log4j

Tokens: ['http', ':', '/', '/', ':', '5000', '/', 'greet', '/', '%', '7', '##b', '%', '7', '##b', '##get', '_', 'flashed', '_', 'messages', '.', '_', '_', 'global', '##s', '_', '_', '.', '_', '_', 'built', '##ins', '_', '_', '.', 'print', '(', "'", 'running', '%', '20', '##pa', '##yl', '##oa', '##d', '!', "'", ')', '%', '7'

In [57]:
# Step 2: Tokenize the payload texts and aggregate tokens
tokens_list = []
for _, row in df.iterrows():
    payload = row.get("payload")
    if pd.notnull(payload):
        tokens = tokenizer.tokenize(payload)  # Tokenize the cleaned payload
        tokens_list.extend(tokens)  # Add tokens to the list

# Step 3: Generate unique vocabulary
vocab = sorted(set(tokens_list))

# Step 4: Remove existing vocab file if it exists
vocab_file = './Datasets/vocab.txt'
if os.path.exists(vocab_file):
    os.remove(vocab_file)

# Step 5: Save the new vocabulary to a file
with open(vocab_file, 'w', encoding='utf-8') as vocab_outfile:
    vocab_outfile.write("\n".join(vocab))  # Write each token on a new line

# Print a preview of the vocabulary
print("\nVocabulary Example:")
print(vocab[:50])  # Show first 50 tokens for clarity


Vocabulary Example:
['!', '##0', '##00', '##01', '##1', '##10', '##100', '##11', '##12', '##13', '##14', '##15', '##16', '##17', '##18', '##19', '##2', '##20', '##21', '##22', '##23', '##24', '##25', '##26', '##27', '##28', '##29', '##3', '##30', '##31', '##32', '##33', '##34', '##35', '##36', '##37', '##38', '##39', '##4', '##40', '##41', '##42', '##43', '##44', '##45', '##46', '##47', '##48', '##49', '##5']
